# Logistic regression tutorial

We train **binary** `mlpackage.supervised_learning.LogisticRegression` (batch gradient descent on cross-entropy) on sklearn's **Wisconsin Breast Cancer** data: **569** samples, **30** numeric features, labels **malignant** vs **benign** (encoded 0/1). Feature **standardization** (fit on train only) helps stable gradient steps.

## Prerequisites and goals

**You will**
- Load a two-class problem with continuous features.
- Split train/test with **stratification** so both classes appear in each fold.
- Standardize using **training** mean/variance, then fit logistic regression.
- Evaluate **accuracy**, inspect **predicted probabilities**, and compare **iteration counts** on the same split.
- Plot a **linear decision boundary** in 2D with the **test** points overlaid (using a small 2-feature model for the figure).

**Dependencies:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`, `mlpackage`.

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from mlpackage.supervised_learning import LogisticRegression

## Step 1: Load Breast Cancer data

Targets: **0 = malignant**, **1 = benign** (sklearn ordering). The feature matrix is already numeric; we will scale it before optimization.

In [ ]:
cancer = load_breast_cancer()
X = np.asarray(cancer.data, dtype=float)
y = np.asarray(cancer.target, dtype=int)

feature_names = list(cancer.feature_names)
target_names = list(cancer.target_names)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Class names:", {i: target_names[i] for i in range(2)})
print("Label counts:", dict(zip(*np.unique(y, return_counts=True))))
print("\nFirst feature row (raw):", np.round(X[0], 3))

## Step 2: Stratified train/test split

With two classes, stratification keeps minority/majority rates similar in train and test (here the classes are only mildly imbalanced, but the habit is good practice).

In [ ]:
RANDOM_STATE = 42
TEST_FRACTION = 0.30

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_FRACTION,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train:", X_train.shape[0], "  Test:", X_test.shape[0])
print("Train counts:", dict(zip(*np.unique(y_train, return_counts=True))))
print("Test counts :", dict(zip(*np.unique(y_test, return_counts=True))))

## Step 3: Standardize features (train-only fit)

Gradient descent can converge faster when inputs share comparable scales. **`StandardScaler`** learns mean/variance from **`X_train`** only.

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Standardized train shape:", X_train_s.shape)

## Step 4: Fit `LogisticRegression`

Defaults (`learning_rate=0.01`, `n_iterations=1000`) often work on scaled tabular data; adjust if loss plateaus slowly.

### Model and loss

For binary $y_i \in \{0,1\}$, with scores $z_i = \mathbf{w}^\top \mathbf{x}_i + b$ and probabilities $p_i = \sigma(z_i)$,

$$
\sigma(z) = \frac{1}{1+e^{-z}}, \qquad
\mathcal{L} = -\frac{1}{n}\sum_{i=1}^{n} \Bigl[ y_i \log p_i + (1-y_i)\log(1-p_i) \Bigr].
$$

This is negative log-likelihood for independent Bernoulli outcomes (**binary cross-entropy**). Gradients (batch averages) are

$$
\frac{\partial \mathcal{L}}{\partial \mathbf{w}} = \frac{1}{n}\mathbf{X}^\top(\mathbf{p}-\mathbf{y}), \qquad
\frac{\partial \mathcal{L}}{\partial b} = \frac{1}{n}\sum_i (p_i - y_i).
$$

Each iteration does $\mathbf{w} \leftarrow \mathbf{w} - \eta \nabla_{\mathbf{w}}\mathcal{L}$ and $b \leftarrow b - \eta \, \partial\mathcal{L}/\partial b$ with learning rate $\eta$. The **logit** $\operatorname{logit}(p)=\log\frac{p}{1-p}$ equals the linear score, so this is a GLM with logit link. The loss is **convex** in $(\mathbf{w},b)$; gradient descent converges to a global minimum under typical conditions.



In [ ]:
LEARNING_RATE = 0.1
N_ITER = 2000

clf = LogisticRegression(learning_rate=LEARNING_RATE, n_iterations=N_ITER)
clf.fit(X_train_s, y_train.astype(float))

print("Bias b:", round(clf.bias, 4))
w = pd.Series(clf.weights, index=feature_names, name="weights").sort_values(key=abs, ascending=False)
display(w.head(10).round(4))

## Step 5: Predict classes and probabilities on the test set

**`predict_probability`** returns $\hat{p}_i = P(y_i=1\mid \mathbf{x}_i) = \sigma(\mathbf{w}^\top \mathbf{x}_i + b)$ per row.

**`predict`** applies the usual **0.5** threshold on $\hat{p}_i$:

$$
\hat{y}_i = \mathbf{1}\{\hat{p}_i \ge 0.5\}
$$

(class **1** if $\hat{p}_i \ge 0.5$, else **0** in this implementation).


In [ ]:
p_test = clf.predict_probability(X_test_s)
y_pred = clf.predict(X_test_s)

n_show = 12
preview = pd.DataFrame(
    {
        "y_true": [target_names[t] for t in y_test[:n_show]],
        "P(benign)": np.round(p_test[:n_show], 4),
        "y_pred": [target_names[t] for t in y_pred[:n_show]],
    }
)
display(preview)

## Step 6: Accuracy and confusion-style counts

`score` is **accuracy**; we also show per-class **true positive** style tallies for the 2×2 structure.

In [ ]:
print(f"Train accuracy: {clf.score(X_train_s, y_train):.4f}")
print(f"Test accuracy : {clf.score(X_test_s, y_test):.4f}")

for name, y_t, y_p in [
    ("Test", y_test, y_pred),
]:
    for true_label in (0, 1):
        mask = y_t == true_label
        correct = int(np.sum((y_p == y_t) & mask))
        total = int(np.sum(mask))
        print(
            f"{name} true {target_names[true_label]}: {correct}/{total} correct"
        )

## Step 7: Decision boundary in 2D (test set overlaid)

The fitted model in Steps 4–6 uses **all 30** standardized features. A decision surface in \(\mathbb{R}^{30}\) cannot be drawn directly. Here we **refit** `LogisticRegression` on **only two** columns — **mean radius** and **mean texture** — so we can plot the **linear** boundary \(\mathbf{w}^\top\mathbf{x}+b=0\), shade predicted regions (class 0 vs 1), and overlay **test** samples.

**Markers:** class 0 = red squares, class 1 = blue crosses (style matches common lecture slides). Misclassified test points appear in the wrong shaded region.

In [ ]:
from pathlib import Path

i, j = 0, 1
X_train_2d = X_train_s[:, [i, j]]
X_test_2d = X_test_s[:, [i, j]]

viz_clf = LogisticRegression(learning_rate=LEARNING_RATE, n_iterations=N_ITER)
viz_clf.fit(X_train_2d, y_train.astype(float))

h = 0.02
pad = 0.6
x_min = float(min(X_train_2d[:, 0].min(), X_test_2d[:, 0].min()) - pad)
x_max = float(max(X_train_2d[:, 0].max(), X_test_2d[:, 0].max()) + pad)
y_min = float(min(X_train_2d[:, 1].min(), X_test_2d[:, 1].min()) - pad)
y_max = float(max(X_train_2d[:, 1].max(), X_test_2d[:, 1].max()) + pad)

xx, yy = np.meshgrid(
    np.arange(x_min, x_max, h),
    np.arange(y_min, y_max, h),
)
grid_X = np.c_[xx.ravel(), yy.ravel()]
Z = viz_clf.predict(grid_X).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(
    xx,
    yy,
    Z,
    levels=[-0.5, 0.5, 1.5],
    colors=["#ffcccc", "#cce5ff"],
    alpha=0.95,
)

w2 = viz_clf.weights
b2 = viz_clf.bias
xs_line = np.linspace(x_min, x_max, 200)
if abs(w2[1]) > 1e-10:
    ys_line = -(w2[0] * xs_line + b2) / w2[1]
    plt.plot(xs_line, ys_line, "k-", linewidth=2.0, label="Decision boundary")
else:
    x_vert = -b2 / w2[0]
    plt.axvline(x_vert, color="k", linewidth=2.0, label="Decision boundary")

m0 = y_test == 0
m1 = y_test == 1
plt.scatter(
    X_test_2d[m0, 0],
    X_test_2d[m0, 1],
    c="red",
    marker="s",
    s=55,
    edgecolors="black",
    linewidths=0.6,
    label=f"{target_names[0]} (class 0, test)",
    zorder=5,
)
plt.scatter(
    X_test_2d[m1, 0],
    X_test_2d[m1, 1],
    c="blue",
    marker="x",
    s=85,
    linewidths=1.6,
    label=f"{target_names[1]} (class 1, test)",
    zorder=5,
)

plt.xlabel(f"Feature 1 — {feature_names[i]} [standardized]")
plt.ylabel(f"Feature 2 — {feature_names[j]} [standardized]")
plt.title("Logistic Regression Decision Boundary (Test Set)")
plt.legend(loc="upper right", fontsize=9)
plt.xlim(x_min, x_max)
plt.ylim(y_min, y_max)
plt.tight_layout()

_nb_here = Path("logistic_regression_tutorial.ipynb")
if _nb_here.is_file():
    _plot_path = _nb_here.with_name("breast_cancer_logistic_decision_boundary.png")
else:
    _plot_path = Path(
        "examples/supervised_learning/logistic_regression/breast_cancer_logistic_decision_boundary.png"
    )
    _plot_path.parent.mkdir(parents=True, exist_ok=True)

plt.savefig(_plot_path, dpi=150, bbox_inches="tight")
print(f"Saved figure to: {_plot_path.resolve()}")
print("2-feature test accuracy (viz model):", round(viz_clf.score(X_test_2d, y_test), 4))
plt.show()

## Step 8: Compare `n_iterations` (same split, same `learning_rate`)

More iterations can improve fit up to a point; if learning rate is too large, training can become unstable. Here we only vary **`n_iterations`**.

In [ ]:
iters = [200, 500, 1000, 2000, 5000]
rows = []

for n in iters:
    m = LogisticRegression(learning_rate=LEARNING_RATE, n_iterations=n)
    m.fit(X_train_s, y_train.astype(float))
    rows.append(
        {
            "n_iterations": n,
            "train_acc": m.score(X_train_s, y_train),
            "test_acc": m.score(X_test_s, y_test),
        }
    )

display(pd.DataFrame(rows).round(4))